# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import gc
from sklearn.metrics import f1_score, roc_auc_score, recall_score, precision_score, accuracy_score
import tqdm
import scipy.stats as stats
import pickle
import re

import warnings
warnings.filterwarnings('ignore')

# User-Defined Parameteres

In [ ]:
DATAPATH = 'ACE-AI/Data/Raw Data/Survey/'
PROCDATAPATH = 'ACE-AI/Data/Processed Data/'
SURVEY_ONBOARD = 'SURVEY_ONBOARD_PROCESSED.csv'
SURVEY_COMB_RAW = 'SURVEY_COMBINED_UNPROCESSED.csv'

# Load Data

In [ ]:
survey_onboard = pd.read_csv(DATAPATH + SURVEY_ONBOARD)
survey_comb_raw = pd.read_csv(DATAPATH + SURVEY_COMB_RAW)  # Post-consultation survey

# Doctor Background

## Age

In [ ]:
survey_onboard.groupby(['Q18'])['DOCTOR'].nunique().reset_index()

## Challenge in CDM and CDA

In [ ]:
import itertools

tmp = itertools.chain(*survey_onboard['Q4'].str.split(';'))
figdata = pd.Series(tmp).value_counts().reset_index().rename(columns={'index':'Reason', 0:'Count'})

# Without Access to ACE-AI

## Specific-Defined Parameters

In [ ]:
reason_options = [
    "Hyperlipidemia, Hypertension, Diabetes",
    "Acute upper respiratory tract infection or fever",
    "Aches and pains, minor wounds or minor sprains and fracture",
    "Abdominal pain; indigestion; gastric discomfort",
    "Dermatological conditions",
    "Mental health conditions",
    "Health screening",
    "Medication refill",
    "Referral",
    "Others",
]

chronic_condition_options = [
    "No existing conditions",
    "Unknown",
    "Diabetes Mellitus/ Pre-diabetes",
    "Hypertension",
    "Hyperlipidemia (Lipid Disorders)",
    "Stroke or Ischaemic Heart Disease",
    "Allergic Rhinitis or Asthma",
    "Chronic Obstructive Pulmonary Disease (COPD)",
    "Mental health conditions",
    "Dementia",
    "Osteoarthritis",
    "Benign Prostatic Hyperplasia",
    "Parkinson's Disease",
    "Chronic Kidney Disease (Nephrosis/Nephritis)",
    "Epilepsy",
    "Osteoporosis",
    "Chronic skin conditions",
    "Rheumatoid Arthritis",
    "Gout",
    "Chronic liver conditions",
    "Others",
]

no_assessment_reason_options = [
    "Time constraints",
    "Not patient's concern",
    "Risk assessment tools difficult to interpret",
    "Problems with recording and retrieval of NEHR data",
    "Lack of referral resources",
    "Lack of patient adherence/compliance",
    "Others",
]

risk_assessment_disease_options = [
    "Diabetes Mellitus",
    "Hypertension",
    "Dyslipidaemia",
    "Heart Disease",
    "Renal Disease",
    "Osteoarthritis",
    "Osteoporosis",
    "Stroke",
    "Others",
]

question_config = [
    {
        'question': "Qn1. What is the patient's gender?",
        'column': 'GENDER',
        'order': ['Female', 'Male'],
    },
    {
        'question': "Qn2. What is the patient's age group?",
        'column': 'AGEGROUP',
        'order': ['30-39 years old', '40-49 years old', '50-65 years old', '>65 years old'],
    },
    {
        'question': "Qn3. Please list the main reason for the patient's visit:",
        'column': 'Q1',
        'type': 'multi',
        'options': reason_options,
    },
    {
        'question': "Qn4. Please list the existing chronic conditions of the patient:",
        'column': 'Q2',
        'type': 'multi',
        'options': chronic_condition_options,
    },
    {
        'question': "Qn5. Did you do risk assessment on chronic diseases for this patient during the consultation",
        'column': 'Q22',
        'order': ['No', 'Yes'],
    },
    {
        'question': 'Qn6. Reply "No" to Qn5: Why didn’t you do so?',
        'column': 'Q23',
        'type': 'multi',
        'options': no_assessment_reason_options,
        'filter_column': 'Q22',
        'filter_value': ['No'],
    },
    {
        'question': 'Qn7. Reply "No" to Qn5: If this patient\'s health risks can be immediately available, would you conduct risk assessment on chronic diseases for him/her?',
        'column': 'Q24',
        'order': ['No', 'Yes'],
        'filter_column': 'Q22',
        'filter_value': ['No'],
    },
    {
        'question': 'Qn8. Reply "Yes" to Qn5: How much time did you spend on evaluating the disease risks for this patient?',
        'column': 'Q25',
        'order': ['1-5 minutes', '6-10 minutes', '11-15 minutes', '16-20 minutes', '>20 minutes'],
        'filter_column': 'Q22',
        'filter_value': ['Yes'],
    },
    {
        'question': 'Qn9. Reply "Yes" to Qn5: What diseases did you conduct risk assessments for this patient?',
        'column': 'Q26',
        'type': 'multi',
        'options': risk_assessment_disease_options,
        'filter_column': 'Q22',
        'filter_value': ['Yes'],
    },
    {
        'question': "Qn10. Did you look through NEHR data of the patient for this visit?",
        'column': 'Q20',
        'order': ['No', 'Yes'],
    },
]

## Specific-Defined Functions

In [ ]:
def _norm(s):
    result = re.sub(r"\s+", " ", str(s)).strip().lower()
    result = re.sub(r"’", "'", result)
    return result

def parse_multi(resp, options, others_label="Others"):
    """Return (flags dict, others free-text, unmatched residual)."""
    flags = {o: 0 for o in options}
    if pd.isna(resp) or not str(resp).strip():
        return flags, None, ""

    residual = _norm(resp)
    fixed = [o for o in options if o != others_label]

    # strip longest option first so e.g. "Diabetes" can't eat "Diabetes, insulin-treated"
    for opt in sorted(fixed, key=len, reverse=True):
        key = _norm(opt)
        if key in residual:
            flags[opt] = 1
            residual = residual.replace(key, " ")

    other_txt = None
    if others_label in options:
        pat = rf"{_norm(others_label).rstrip('s')}s?\s*:?\s*(.*)"
        m = re.search(pat, residual, flags=re.DOTALL)
        flags[others_label] = int(bool(m))
        if m:
            other_txt = m.group(1).strip(" ;,.") or None
            residual = re.sub(pat, " ", residual, flags=re.DOTALL)

    residual = re.sub(r"[\s;,]+", " ", residual).strip()
    return flags, other_txt, residual

In [ ]:
def build_survey_table(df, config, return_qc=False):
    """Summarise single- and multi-select survey questions into one flat table.

    Config keys per question:
        question       display label (required)
        column         dataframe column (required)
        type           'single' (default) or 'multi'
        options        list of canonical options (required when type='multi')
        others_label   free-text option label, default 'Others'
        order          display order of values (optional)
        filter_column  restrict to rows where this column == filter_value
        filter_value
    """
    rows, qc = [], {}

    for q in config:
        col          = q['column']
        qtype        = q.get('type', 'single')
        order        = q.get('order')
        filter_col   = q.get('filter_column')
        filter_value = q.get('filter_value')

        subset = df[df[filter_col].isin(filter_value)] if filter_col is not None else df

        if qtype == 'multi':
            options = q['options']
            others  = q.get('others_label', 'Others')

            parsed  = [parse_multi(r, options, others) for r in subset[col]]
            flag_df = pd.DataFrame([p[0] for p in parsed],
                                   index=subset.index,
                                   columns=options).fillna(0).astype(int)

            n_sel = flag_df.sum(axis=1)
            n = subset.shape[0]

            cnt = flag_df.sum().reindex(order or options, fill_value=0)
            pct = (cnt / n * 100) if n else cnt.astype(float)

            label = f"{q['question']}  (multi-select; % of respondents)"

            qc_key = col if col not in qc else f"{col} | {q['question'][:40]}"
            qc[qc_key] = pd.DataFrame({
                'response':    subset[col],
                'others_text': [p[1] for p in parsed],
                'unmatched':   [p[2] for p in parsed],
                'n_selected':  n_sel,
            })

        else:
            cnt = subset[col].value_counts(dropna=False)
            pct = subset[col].value_counts(dropna=False, normalize=True) * 100
            n   = subset.shape[0]

            if order:
                extra = [v for v in cnt.index if pd.isna(v) or v not in order]
                cnt = cnt.reindex(list(order) + extra, fill_value=0)
                pct = pct.reindex(list(order) + extra, fill_value=0.0)
            else:
                pct = pct.sort_values(ascending=False)
                cnt = cnt.reindex(pct.index)

            label = q['question']

        for i, (value, count) in enumerate(cnt.items()):
            rows.append({
                'Survey Question': label if i == 0 else '',
                'Value': value if pd.notna(value) else 'Missing',
                'Count': int(count) if pd.notna(count) else 0,
                'Percentage': f'{pct.iloc[i]:.1f}%' if pd.notna(pct.iloc[i]) else '0.0%',
                'N': n if i == 0 else '',
            })

    table = pd.DataFrame(rows)
    return (table, qc) if return_qc else table

## Run for All

In [ ]:
survey_noaccess_raw = survey_comb_raw[survey_comb_raw['ACCESS ACEAI']==False].copy()
print(survey_noaccess_raw.shape)

survey_table = build_survey_table(survey_noaccess_raw, question_config)

survey_table.to_csv(f'{PROCDATAPATH}survey_summary_table_noaccess.csv', index=False)
print('Saved survey_summary_table_noaccess.csv')

# With Access to ACE-AI

## Specific-Defined Parameters

In [ ]:
likert_order = ['Strongly agree', 'Agree', 'Neutral', 'Disagree', 'Strongly disagree']

reason_options = [
    "Hyperlipidemia, Hypertension, Diabetes",
    "Acute upper respiratory tract infection or fever",
    "Aches and pains, minor wounds or minor sprains and fracture",
    "Abdominal pain; indigestion; gastric discomfort",
    "Dermatological conditions",
    "Mental health conditions",
    "Health screening",
    "Medication refill",
    "Referral",
    "Others",
]

chronic_condition_options = [
    "No existing conditions",
    "Unknown",
    "Diabetes Mellitus/ Pre-diabetes",
    "Hypertension",
    "Hyperlipidemia (Lipid Disorders)",
    "Stroke or Ischaemic Heart Disease",
    "Allergic Rhinitis or Asthma",
    "Chronic Obstructive Pulmonary Disease (COPD)",
    "Mental health conditions",
    "Dementia",
    "Osteoarthritis",
    "Benign Prostatic Hyperplasia",
    "Parkinson's Disease",
    "Chronic Kidney Disease (Nephrosis/Nephritis)",
    "Epilepsy",
    "Osteoporosis",
    "Chronic skin conditions",
    "Rheumatoid Arthritis",
    "Gout",
    "Chronic liver conditions",
    "Others",
]

q6_neutral_reason_options = [
    "The patient does not have any existing chronic conditions based on ACE-AI's assessment",
    "I already know some of the existing diseases that ACE-AI informed",
    "Others",
]

q6_disagree_reason_options = [
    "ACE-AI failed to inform one/some of the existing conditions within the listed 8 diseases",
    "ACE-AI incorrectly informed some existing conditions",
    "Others",
]

q9_neutral_reason_options = [
    "The patient does not have any disease risks based on ACE-AI's assessment (i.e. all normal risk)",
    "I already know some of disease risks that ACE-AI informed",
    "Others",
]

disease_options = [
    "Diabetes",
    "Hypertension",
    "Dyslipidaemia",
    "Heart Disease",
    "Renal Disease",
    "Osteoarthritis",
    "Osteoporosis",
    "Stroke",
]

no_assessment_reason_options = [
    "Time constraints",
    "Not patient's concern",
    "Risk assessment tools difficult to interpret",
    "Problems with recording and retrieval of NEHR data",
    "Lack of referral resources",
    "Lack of patient adherence/compliance",
    "Others",
]

q14_neutral_reason_options = [
    "ACE-AI only helps me understand some disease risks",
    "ACE-AI did not display enough risk factors",
    "Others",
]

q17_neutral_reason_options = [
    'The patient does not have any disease risks based on ACE-AI’s assessment (i.e. all normal risk).',
    'No health recommendations were made because chronic disease management was not the main reason of this visit.',
    'ACE-AI only helps me make recommendations for some diseases',
    'Others',
]

q17_disagree_reason_options = [
    "Insufficient risk factors were provided",
    "Risk factors shown are not modifiable",
    "Others",
]

q20_neutral_reason_options = [
    "ACE-AI only supported for some diseases",
    "No chronic disease risk assessment and management were done as it was not the main purpose of this visit",
    "Others",
]

question_config = [
    {
        'question': "Qn2. What is the patient's gender?",
        'column': 'GENDER',
        'order': ['Female', 'Male'],
    },
    {
        'question': "Qn3. What is the patient's age group?",
        'column': 'AGEGROUP',
        'order': ['30-39 years old', '40-49 years old', '50-65 years old', '>65 years old'],
    },
    {
        'question': "Qn4. Please list the main reason for the patient's visit:",
        'column': 'Q1',
        'type': 'multi',
        'options': reason_options,
    },
    {
        'question': "Qn5. Please list the existing chronic conditions of the patient:",
        'column': 'Q2',
        'type': 'multi',
        'options': chronic_condition_options,
    },
    {
        'question': "Qn6. ACE-AI informed me of this patient's existing chronic conditions that I might not have noticed otherwise.",
        'column': 'Q3',
        'order': likert_order,
    },
    {
        'question': 'Qn7. Reply "Neutral" to Qn6: Please list the reason',
        'column': 'Q4',
        'type': 'multi',
        'options': q6_neutral_reason_options,
        'filter_column': 'Q3',
        'filter_value': ['Neutral'],
    },
    {
        'question': 'Qn8. Reply "Disagree" or "Strongly disagree" to Qn6: Please list the reason',
        'column': 'Q5',
        'type': 'multi',
        'options': q6_disagree_reason_options,
        'filter_column': 'Q3',
        'filter_value': ['Disagree', 'Strongly disagree'],
    },
    {
        'question': "Qn9. ACE-AI informed me of this patient's disease risks that I might not have noticed otherwise.",
        'column': 'Q6',
        'order': likert_order,
    },
    {
        'question': 'Qn10. Reply "Neutral" to Qn9: Please list the reason',
        'column': 'Q7',
        'type': 'multi',
        'options': q9_neutral_reason_options,
        'filter_column': 'Q6',
        'filter_value': ['Neutral'],
    },
    {
        'question': 'Qn11. Reply "Disagree" or "Strongly disagree" to Qn9: Which disease risk do you disagree with?',
        'column': 'Q8',
        'type': 'multi',
        'options': disease_options,
        'filter_column': 'Q6',
        'filter_value': ['Disagree', 'Strongly disagree'],
    },
    {
        'question': "Qn12. I would conduct a chronic disease risk assessment for this patient even if ACE-AI was not available to me.",
        'column': 'Q9',
        'order': likert_order,
    },
    {
        'question': 'Qn13. Reply "Neutral", "Disagree" or "Strongly disagree" to Qn12: Why won\'t you do so?',
        'column': 'Q10',
        'type': 'multi',
        'options': no_assessment_reason_options,
        'filter_column': 'Q9',
        'filter_value': ['Neutral', 'Disagree', 'Strongly disagree'],
    },
    {
        'question': "Qn14. The risk rationale of ACE-AI helps me understand disease risks for this patient.",
        'column': 'Q11',
        'order': likert_order,
    },
    {
        'question': 'Qn15. Reply "Neutral" to Qn14: Please list the reason',
        'column': 'Q12',
        'type': 'multi',
        'options': q14_neutral_reason_options,
        'filter_column': 'Q11',
        'filter_value': ['Neutral'],
    },
    {
        'question': "Qn17. ACE-AI helps me make health recommendations for this patient.",
        'column': 'Q14',
        'order': likert_order,
    },
    {
        'question': 'Qn18. Reply "Neutral" to Qn17: Please list the reason',
        'column': 'Q15',
        'type': 'multi',
        'options': q17_neutral_reason_options,
        'filter_column': 'Q14',
        'filter_value': ['Neutral'],
    },
    {
        'question': 'Qn19. Reply "Disagree" or "Strongly disagree" to Qn17: Please list the reason',
        'column': 'Q16',
        'type': 'multi',
        'options': q17_disagree_reason_options,
        'filter_column': 'Q14',
        'filter_value': ['Disagree', 'Strongly disagree'],
    },
    {
        'question': "Qn20. ACE-AI supported my ability to conduct chronic disease risk assessment and management for this patient.",
        'column': 'Q17',
        'order': likert_order,
    },
    {
        'question': 'Qn21. Reply "Neutral" to Qn20: Please list the reason',
        'column': 'Q18',
        'type': 'multi',
        'options': q20_neutral_reason_options,
        'filter_column': 'Q17',
        'filter_value': ['Neutral'],
    },
    {
        'question': "Qn23. Did you look through NEHR data of the patient for this visit?",
        'column': 'Q20',
        'order': ['No', 'Yes'],
    },
]

## Run for All

In [ ]:
survey_access_raw = survey_comb_raw[survey_comb_raw['ACCESS ACEAI']==True].copy()
print(survey_access_raw.shape)

survey_table = build_survey_table(survey_access_raw, question_config)

survey_table.to_csv(f'{PROCDATAPATH}survey_summary_table_access.csv', index=False)
print('Saved survey_summary_table_access.csv')

# End